In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

import os
import datetime
import pickle
import itertools

import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Image

from rdkit import Chem
from collections import Counter

from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.layers import Layer, LayerNormalization, Dense
from sklearn.model_selection import ParameterGrid


tf.keras.backend.clear_session()

# Load Data

In [ ]:
# Load training data, val and test data
file_path = 'train.pkl'
with open(file_path, 'rb') as file:
    train_df = pickle.load(file)
# Load validation data
file_path = 'validation_small_enantiomers_stable_full_screen_docking_MOL_margin3_49878_10368_5184.pkl'
with open(file_path, 'rb') as file:
    val_df = pickle.load(file)
file_path = 'test_small_enantiomers_stable_full_screen_docking_MOL_margin3_50571_10368_5184.pkl'
with open(file_path, 'rb') as file:
    test_df = pickle.load(file)


# Get atoms in list

In [ ]:

def atomic_number_to_symbol(atomic_number):
    periodic_table = {
        1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F',
        15: 'P', 16: 'S', 17: 'Cl', 35: 'Br', 53: 'I'
    }
    return periodic_table.get(atomic_number, 'Unknown')

# Define dataset paths
datasets = {
    'train': 'train.pkl',
    'val': 'validation_small_enantiomers_stable_full_screen_docking_MOL_margin3_49878_10368_5184.pkl',
    'test': 'test_small_enantiomers_stable_full_screen_docking_MOL_margin3_50571_10368_5184.pkl'
}

mollists = {}

for dataset_name, file_path in datasets.items():
    with open(file_path, 'rb') as file:
        df = pickle.load(file)
    
    mollist = []
    for mol in df['rdkit_mol_cistrans_stereo']:
        molecule = []
        if mol is not None and isinstance(mol, Chem.Mol):
            if mol.GetNumConformers() > 0:
                conf = mol.GetConformer(0)
                for atom in mol.GetAtoms():
                    atomic_number = atom.GetAtomicNum()
                    atom_type = atomic_number_to_symbol(atomic_number)
                    pos = conf.GetAtomPosition(atom.GetIdx())
                    coordinates = np.array([pos.x, pos.y, pos.z])
                    molecule.append((atom_type, coordinates))
        mollist.append(molecule)
    
    mollists[dataset_name] = mollist



In [ ]:
train_mollist = mollists['train']
val_mollist = mollists['val']
test_mollist = mollists['test']

# Train test split

In [ ]:
%%time
X_train = train_mollist    
y_train = train_df['top_score']

X_val = val_mollist             
y_val = val_df['top_score']    

X_test = test_mollist          
y_test = test_df['top_score']  

In [ ]:

def speciesmap(atom_type):
    atom_to_number = {
        'H': 1,   # Hydrogen
        'C': 6,   # Carbon
        'N': 7,   # Nitrogen
        'O': 8,   # Oxygen
        'F': 9,   # Fluorine
        'P': 15,  # Phosphorus
        'S': 16,  # Sulfur
        'Cl': 17, # Chlorine
        'Br': 35, # Bromine
        'I': 53   # Iodine
    }
    return np.array([atom_to_number.get(atom_type, 0)])  # Returns 0 if atom type is not recognized

In [ ]:
%%time
from views_train import load_chiral_data
from views_val import load_chiral_data
from views_test import load_chiral_data

ws_train, vs_train, Natoms_train, Nviews_train = load_chiral_data(train_mollist, speciesmap, setNatoms=29, setNviews=None, carbonbased=False, verbose=0)
chiral_train = [ws_train, vs_train]

# Featurize validation data
ws_val, vs_val, Natoms_val, Nviews_val = load_chiral_data(val_mollist, speciesmap, setNatoms=29, setNviews=29, carbonbased=False, verbose=0)
chiral_val = [ws_val, vs_val]

# Featurize test data
ws_test, vs_test, Natoms_test, Nviews_test = load_chiral_data(test_mollist, speciesmap, setNatoms=29, setNviews=29, carbonbased=False, verbose=0)
chiral_test = [ws_test, vs_test]

In [ ]:
from enantiomer_single_atom import small_views
single_atom_train = small_views(vs_train)
single_atom_val = small_views(vs_val)
single_atom_test = small_views(vs_test)


In [ ]:
%%time
from enantiomer_single_atom import get_embeddings, atom_properties, single_atomic_property_switches
atom_embeddings_train = get_embeddings(single_atom_train, atom_properties, single_atomic_property_switches)
atom_embeddings_val = get_embeddings(single_atom_val, atom_properties, single_atomic_property_switches)
atom_embeddings_test = get_embeddings(single_atom_test, atom_properties, single_atomic_property_switches)

In [ ]:

# Copy data
Xtr = atom_embeddings_train.copy()
Xval = atom_embeddings_val.copy()
Xte = atom_embeddings_test.copy()

# Create masks (nonzero atoms)
mask_tr = (Xtr[..., 0] != 0)
mask_val = (Xval[..., 0] != 0)
mask_te = (Xte[..., 0] != 0)

# Fit scaler 
props_tr = Xtr[..., 4:12][mask_tr]
prop_scaler = StandardScaler().fit(props_tr)

# Apply transform to all three sets
Xtr[..., 4:12][mask_tr]  = prop_scaler.transform(Xtr[..., 4:12][mask_tr])
Xval[..., 4:12][mask_val] = prop_scaler.transform(Xval[..., 4:12][mask_val])
Xte[..., 4:12][mask_te]  = prop_scaler.transform(Xte[..., 4:12][mask_te])


In [ ]:

labelsG_train = y_train 
labelsG_val = y_val     
labelsG_test = y_test   
Ntoxicity = 3  

ws_train, vs_train = ws_train, Xtr 
ws_val, vs_val = ws_val, Xval   
ws_test, vs_test = ws_test, Xte  


dataG_train = [ws_train, vs_train]
dataG_val = [ws_val, vs_val]              
dataG_test = [ws_test, vs_test]

# Compute Coulomb matrix

In [ ]:
def compute_coulomb_matrix_for_view(view):
    """
    Compute Coulomb matrix for a single view.
    
    Parameters:
    - view: numpy array of shape (29, 4) where each row is [atomic_number, x, y, z]
    
    Returns:
    - coulomb_matrix: numpy array of shape (29, 29)
    """
    n_atoms = view.shape[0]
    coulomb_matrix = np.zeros((n_atoms, n_atoms))
    
    for i in range(n_atoms):
        atomic_num_i = view[i, 0]
        if atomic_num_i == 0:
            continue
            
        coords_i = view[i, 1:4]
        
        for j in range(n_atoms):
            atomic_num_j = view[j, 0]
            if atomic_num_j == 0:  # Skip padding atoms
                continue
                
            coords_j = view[j, 1:4]
            
            if i == j:
                # Diagonal element: self-interaction
                coulomb_matrix[i, j] = 0.5 * atomic_num_i ** 2.4
            else:
                # Off-diagonal element: interaction between atoms i and j
                dist = np.linalg.norm(coords_i - coords_j)
                coulomb_matrix[i, j] = atomic_num_i * atomic_num_j / dist
                
    return coulomb_matrix

In [ ]:
def precompute_coulomb_matrices_for_views(views_data):
    """
    Precompute Coulomb matrices for all views in the dataset.
    
    Parameters:
    - views_data: numpy array of shape (batch_size, num_views, num_atoms, 4)
    
    Returns:
    - coulomb_matrices: numpy array of shape (batch_size, num_views, num_atoms, num_atoms)
    """
    batch_size, num_views, num_atoms, _ = views_data.shape
    coulomb_matrices = np.zeros((batch_size, num_views, num_atoms, num_atoms))
    
    for batch_idx in range(batch_size):
        for view_idx in range(num_views):
            view = views_data[batch_idx, view_idx]
            coulomb_matrices[batch_idx, view_idx] = compute_coulomb_matrix_for_view(view)
                
    return coulomb_matrices

In [ ]:
%%time

coulomb_train = precompute_coulomb_matrices_for_views(single_atom_train)
coulomb_val = precompute_coulomb_matrices_for_views(single_atom_val)
coulomb_test = precompute_coulomb_matrices_for_views(single_atom_test)

# Transformer layer

In [ ]:

# =========================
# DOUBLE HEAD ATTENTION
# =========================
class TwoHeadAtomAttention(Layer):
    def __init__(self, d_k=32, d_v=32, **kwargs):
        super(TwoHeadAtomAttention, self).__init__(**kwargs)
        self.d_k = d_k
        self.d_v = d_v
        self.norm = LayerNormalization(axis=-1, epsilon=1e-6)

    def build(self, input_shape):
        self.num_views = input_shape[1]
        self.num_atoms = input_shape[2]
        self.feature_dim = input_shape[3]

        # ===== HEAD 1 =====
        self.W_q1 = self.add_weight(shape=(self.feature_dim, self.d_k),
                                   initializer='glorot_uniform', trainable=True)
        self.W_k1 = self.add_weight(shape=(self.feature_dim, self.d_k),
                                   initializer='glorot_uniform', trainable=True)
        self.W_v1 = self.add_weight(shape=(self.feature_dim, self.d_v),
                                   initializer='glorot_uniform', trainable=True)

        # ===== HEAD 2 =====
        self.W_q2 = self.add_weight(shape=(self.feature_dim, self.d_k),
                                   initializer='glorot_uniform', trainable=True)
        self.W_k2 = self.add_weight(shape=(self.feature_dim, self.d_k),
                                   initializer='glorot_uniform', trainable=True)
        self.W_v2 = self.add_weight(shape=(self.feature_dim, self.d_v),
                                   initializer='glorot_uniform', trainable=True)

        # Output projection
        self.W_o = self.add_weight(shape=(2 * self.d_v, self.feature_dim),
                                  initializer='glorot_uniform', trainable=True)

        # Coulomb scaling
        self.alpha = self.add_weight(shape=(),
                                     initializer=tf.keras.initializers.Constant(0.1),
                                     trainable=True)

    def attention_head(self, x, W_q, W_k, W_v, coulomb_matrix):
        Q = tf.matmul(x, W_q)
        K = tf.matmul(x, W_k)
        V = tf.matmul(x, W_v)

        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(tf.cast(self.d_k, tf.float32))

        
        row_sums_before = tf.reduce_sum(scores, axis=2)
        zero_rows = tf.equal(row_sums_before, 0.0)

        # Add Coulomb bias
        if coulomb_matrix is not None:
            coulomb_matrix = tf.cast(coulomb_matrix, scores.dtype)
            scores += self.alpha * coulomb_matrix

        # Softmax
        attn = tf.nn.softmax(scores, axis=-1)

        
        zero_rows = tf.expand_dims(zero_rows, axis=-1)
        attn = tf.where(zero_rows, tf.zeros_like(attn), attn)

        output = tf.matmul(attn, V)
        return output

    def call(self, inputs, coulomb_matrix=None):
        batch_size = tf.shape(inputs)[0]

        # Flatten (batch, views)
        x = tf.reshape(inputs, [-1, self.num_atoms, self.feature_dim])

        if coulomb_matrix is not None:
            coulomb_matrix = tf.reshape(
                coulomb_matrix,
                [-1, self.num_atoms, self.num_atoms]
            )

        # Two heads
        out1 = self.attention_head(x, self.W_q1, self.W_k1, self.W_v1, coulomb_matrix)
        out2 = self.attention_head(x, self.W_q2, self.W_k2, self.W_v2, coulomb_matrix)

        # Concatenate heads
        concat = tf.concat([out1, out2], axis=-1)

        # Project back
        projected = tf.matmul(concat, self.W_o)

        # Residual + Norm
        out = self.norm(projected + x)

        # Reshape back
        out = tf.reshape(out, [batch_size, self.num_views, self.num_atoms, self.feature_dim])

        return out


# =========================
# FEED FORWARD
# =========================
class FeedForward(Layer):
    def __init__(self, d_model=20, d_ff=64, **kwargs):
        super(FeedForward, self).__init__(**kwargs)
        self.fc1 = Dense(d_ff, activation='relu')
        self.fc2 = Dense(d_model)
        self.norm = LayerNormalization(axis=-1, epsilon=1e-6)

    def call(self, x):
        y = self.fc1(x)
        y = self.fc2(y)
        return self.norm(x + y)


# =========================
# TRANSFORMER BLOCK
# =========================
class TransformerBlock(Layer):
    def __init__(self, d_model=20, d_k=32, d_v=32, d_ff=64, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.attn = TwoHeadAtomAttention(d_k=d_k, d_v=d_v)
        self.ffn = FeedForward(d_model=d_model, d_ff=d_ff)

    def call(self, inputs, coulomb_matrix=None):
        x = self.attn(inputs, coulomb_matrix=coulomb_matrix)
        x = self.ffn(x)
        return x

In [ ]:
# generic dense NN
def multiDense(Nin,Nout,Nhidden,widthhidden=None,kernel_regularizer=None):
    """Construct a basic NN with some dense layers.
    
    :parameter Nin: The number of inputs
    :type Nin: int
    :parameter Nout: The number of outputs
    :type Nout: int
    :parameter Nhidden: The number of hidden layers.
    :type Nhidden: int
    :parameter widthhidden: The width of each hidden layer.
        If left at None, Nin + Nout will be used.
    :parameter kernel_regularizer: the regularizer to use, such as regularizers.l2(0.001)
    :type kernel_regularizer: tensorflow.keras.regularizers.xxx
    :returns: The NN model
    :rtype: keras.Model
    
    """
    if widthhidden is None:
        widthhidden = Nin + Nout
    x = inputs = keras.Input(shape=(Nin,), name='multiDense_input')
    if kernel_regularizer is not None:
        print("Using regularization")
    for i in range(Nhidden):
        x = layers.Dense(widthhidden, activation='relu', kernel_regularizer=kernel_regularizer,name='dense'+str(i))(x)
#        x = layers.Dense(widthhidden, name='dense'+str(i))(x)
#        x = tf.nn.leaky_relu(x, alpha=0.05)
#    outputs = layers.Dense(Nout, activation='linear',name='multiDense_output')(x)
    outputs = layers.Dense(Nout,name='multiDense_output')(x)
    #outputs = tf.nn.leaky_relu(outputs, alpha=0.05)
    return keras.Model(inputs=inputs, outputs=outputs)#, name='multiDense')


def parallelwrapper(Nparallel,basemodel,insteadmax=False):
    """Construct a model that applies a basemodel multiple times and take a weighted sum (or max) of the result.
    
    :parameter Nparallel: The number of times to apply in parallel
    :type Nparallel: int
    :parameter basemodel: a keras.Model inferred to have Nin inputs and Nout outputs.
    :type basemodel: a keras.Model
    :parameter insteadmax: If True, take the max of the results of the basemodel instead of the weighted sum.
        For compatibility, the model is still constructed with weights as inputs, but it ignores them.
    :type insteadmax: Boolean
    :returns: model with inputs shape [(?,Nparallel),(?,Nin,Nparallel)] and outputs shape (?,Nout).
        The first input is the scalar weights in the sum.
    :rtype: keras.Model
    
    Note: We could do a max over the parallel applications instead of or in addition to the weighted sum.
    
    """
    # infer shape of basemodel inputs and outputs
    Nin =  basemodel.inputs[0].shape[1]
    Nout =  basemodel.outputs[0].shape[1]
    
    # Apply basemodel Nparallel times in parallel
    # create main input (?,Nparallel,Nin) 
    parallel_inputs = keras.Input(shape=(Nparallel,Nin), name='parallelwrapper_input0')
    # apply base NN to each parallel slice; outputs (?,Nparallel,Nout)
    if False:
        # original version, stopped working at some tensorflow update
        xb = basemodel(parallel_inputs) # worked in earlier tensorflow
        #xb = tf.map_fn(basemodel,parallel_inputs) # another version that fails
    else:
        # newer version, works but makes summary and graphing cumbersome
        # unstack in the Nparallel directio
        parallel_inputsunstacked = tf.keras.ops.unstack(parallel_inputs, Nparallel, 1)
        # apply base NN to each 
        xbunstacked = [basemodel(x) for x in parallel_inputsunstacked]
        # re-stack
        xb = tf.keras.ops.stack(xbunstacked,axis=1)
    
    # create input scalars for weighted sun (?,Nparallel)
    weight_inputs = keras.Input(shape=(Nparallel,), name='parallelScalars')
    if insteadmax:
        # take max over the Nparallel direction to get (?,1,Nout)
        out = layers.MaxPool1D(pool_size=Nparallel)(xb)
        # reshape to (?,Nout)
        out = layers.Reshape((Nout,))(out)
    else:
        # do a weighted sum over the Nparallel direction to get (?,Nout)
        out = layers.Dot((-2,-1))([xb,weight_inputs])
    
    return keras.Model(inputs=[weight_inputs,parallel_inputs], outputs=out, name='parallelwrapper')



In [ ]:

def init_generator_with_transformer(
    data, labels, baselayers, Nfeatures, endlayers, Nviews,
    num_blocks=3, d_k=32, d_v=32, d_ff=64,
    base_regularizer=None, end_regularizer=None
):
    """
    data: [ws, vs, coulomb] for the *whole dataset* (used to infer shapes)
    """
    # data[1] is vs: (Nmol, Nviews, Natoms, d_model)
    Natoms  = data[1].shape[2]
    d_model = data[1].shape[3]
    flattened_dim = Natoms * d_model
    
    # Inputs
    weight_input = keras.Input(shape=(Nviews,), name='weight_input')              # (batch, Nviews)
    atom_input   = keras.Input(shape=(Nviews, Natoms, d_model), name='atom_input')# (batch, Nviews, Natoms, d_model)
    coulomb_input = keras.Input(
        shape=(Nviews, Natoms, Natoms),
        name='coulomb_input'
    )  # (batch, Nviews, Natoms, Natoms)
    
    # Transformer blocks with Coulomb
    x = atom_input
    for i in range(num_blocks):
        x = TransformerBlock(
            d_model=d_model,
            d_k=d_k,
            d_v=d_v,
            d_ff=d_ff,
            name=f'transformer_block_{i}'
        )(x, coulomb_matrix=coulomb_input)
    
    # Flatten per view
    x = layers.Reshape((Nviews, flattened_dim), name='flatten_views')(x)
    
    # View-wise NN + weighted sum over views
    Gbase = multiDense(flattened_dim, Nfeatures, baselayers, kernel_regularizer=base_regularizer)
    Gpw   = parallelwrapper(Nviews, Gbase, insteadmax=False)
    x = Gpw([weight_input, x])  # (batch, Nfeatures)
    
    # Final regressor
    Gft = multiDense(Nfeatures, 1, endlayers, kernel_regularizer=end_regularizer)
    output = Gft(x)  # (batch, 1)
    
    generator = keras.Model(
        inputs=[weight_input, atom_input, coulomb_input],
        outputs=output,
        name='generator_with_transformer_coulomb'
    )
    return generator


In [ ]:

def get_enantiomer_pairs(df):
    pairs = []
    smiles_groups = df.groupby('SMILES_nostereo')
    
    for smiles, group in smiles_groups:
        if len(group) >= 2:
            score_groups = group.groupby('top_score')
            enantiomer_data = []
            for score, score_group in score_groups:
                enantiomer_data.append({
                    'conformers': score_group.index.tolist(),
                    'top_score': score,
                    'size': len(score_group)
                })
            if len(enantiomer_data) == 2:
                if np.random.random() > 0.5:
                    pairs.append({
                        'enantiomer1_conformers': enantiomer_data[0]['conformers'],
                        'enantiomer2_conformers': enantiomer_data[1]['conformers'],
                        'enantiomer1_score': enantiomer_data[0]['top_score'],
                        'enantiomer2_score': enantiomer_data[1]['top_score'],
                        'smiles_nostereo': smiles
                    })
                else:
                    pairs.append({
                        'enantiomer1_conformers': enantiomer_data[1]['conformers'],
                        'enantiomer2_conformers': enantiomer_data[0]['conformers'],
                        'enantiomer1_score': enantiomer_data[1]['top_score'],
                        'enantiomer2_score': enantiomer_data[0]['top_score'],
                        'smiles_nostereo': smiles
                    })
    return pairs


def create_enantiomer_batch(pairs, data, labels, batch_size=16):
    batch_ws    = []
    batch_vs    = []
    batch_coul  = []
    batch_scores = []
    
    ws, vs, coulomb = data
    n_pairs = len(pairs)
    
    if n_pairs < batch_size // 2:
        selected_pairs = [pairs[i] for i in np.random.choice(len(pairs), size=batch_size//2, replace=True)]
    else:
        selected_pairs = [pairs[i] for i in np.random.choice(len(pairs), size=batch_size//2, replace=False)]
    
    for pair in selected_pairs:
        conf1_idx = np.random.choice(pair['enantiomer1_conformers'])
        conf2_idx = np.random.choice(pair['enantiomer2_conformers'])
        
        batch_ws.extend([ws[conf1_idx], ws[conf2_idx]])
        batch_vs.extend([vs[conf1_idx], vs[conf2_idx]])
        batch_coul.extend([coulomb[conf1_idx], coulomb[conf2_idx]])
        batch_scores.extend([labels[conf1_idx], labels[conf2_idx]])
    
    return (
        np.array(batch_ws),
        np.array(batch_vs),
        np.array(batch_coul),
        np.array(batch_scores).reshape(-1, 1)
    )


def train_with_enantiomer_sampling(
    generator,
    dataG_train, labelsG_train, train_df,
    dataG_val,   labelsG_val,   val_df,
    epochs=1, batch_size=16
):
    enantiomer_pairs_train = get_enantiomer_pairs(train_df)
    if not enantiomer_pairs_train:
        raise ValueError("No enantiomer pairs found in training data!")
    
    best_val_accuracy = 0.0
    best_weights = None
    patience = 15
    patience_counter = 0
    
    num_batches_per_epoch = max(1, len(enantiomer_pairs_train) // (batch_size // 2))
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx in range(num_batches_per_epoch):
            batch_ws, batch_vs, batch_coul, batch_scores = create_enantiomer_batch(
                enantiomer_pairs_train, dataG_train, labelsG_train, batch_size
            )
            loss = generator.train_on_batch(
                [batch_ws, batch_vs, batch_coul],
                batch_scores
            )
            loss = float(loss[0] if isinstance(loss, (list, tuple)) else loss)
            epoch_loss += loss
        
        avg_loss = epoch_loss / num_batches_per_epoch
        
        val_accuracy = evaluate_ranking_accuracy(generator, dataG_val, labelsG_val, val_df)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_weights = generator.get_weights()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    if best_weights is not None:
        generator.set_weights(best_weights)
    return generator, best_val_accuracy


def evaluate_ranking_accuracy(model, data, labels, df, batch_size=16):
    ws, vs, coulomb = data
    all_predictions = []
    
    for i in range(0, len(ws), batch_size):
        batch_ws   = ws[i:i+batch_size]
        batch_vs   = vs[i:i+batch_size]
        batch_coul = coulomb[i:i+batch_size]
        batch_preds = model.predict(
            [batch_ws, batch_vs, batch_coul],
            verbose=0
        )
        all_predictions.extend(batch_preds.flatten())
    
    all_predictions = np.array(all_predictions)
    
    pairs = get_enantiomer_pairs(df)
    correct_rankings = 0
    total_pairs = len(pairs)
    
    for pair in pairs:
        preds1 = [all_predictions[idx] for idx in pair['enantiomer1_conformers']]
        preds2 = [all_predictions[idx] for idx in pair['enantiomer2_conformers']]
        avg_pred1 = np.mean(preds1)
        avg_pred2 = np.mean(preds2)
        true_score1 = pair['enantiomer1_score']
        true_score2 = pair['enantiomer2_score']
        pred_ranking = avg_pred1 < avg_pred2
        true_ranking = true_score1 < true_score2
        if pred_ranking == true_ranking:
            correct_rankings += 1
    
    accuracy = correct_rankings / total_pairs if total_pairs > 0 else 0.0
    return accuracy

In [ ]:
%%time


# ============================
# Prepare data
# ============================
dataG_train = [
    ws_train.astype('float32'),
    Xtr.astype('float32'),
    coulomb_train.astype('float32')
]

dataG_val = [
    ws_val.astype('float32'),
    Xval.astype('float32'),
    coulomb_val.astype('float32')
]

dataG_test = [
    ws_test.astype('float32'),
    Xte.astype('float32'),
    coulomb_test.astype('float32')
]

# ============================
# Fixed params
# ============================
Nviews = 29

# ============================
# Grid
# ============================
param_grid = {
    'num_blocks': [3],
    'd_k': [32],
    'd_v': [32],
    'd_ff': [64],
    'baselayers': [3],
    'endlayers': [5],
    'Nfeatures': [15],
    'learning_rate': [5e-4],
    'batch_size': [16]
}

grid = list(ParameterGrid(param_grid))
results = []

# ============================
# Grid Search
# ============================
for i, params in enumerate(grid):

    print(f"\n===== RUN {i+1}/{len(grid)} =====")
    print(params)

    baselayers = params['baselayers']
    endlayers  = params['endlayers']
    num_blocks = params['num_blocks']
    d_k        = params['d_k']
    d_v        = params['d_v']
    d_ff       = params['d_ff']
    Nfeatures  = params['Nfeatures']
    lr         = params['learning_rate']
    batch_size = params['batch_size']

    base_regularizer = None
    end_regularizer  = None

    # ============================
    # Build model
    # ============================
    generator = init_generator_with_transformer(
        dataG_train, labelsG_train,
        baselayers, Nfeatures, endlayers, Nviews,
        num_blocks=num_blocks,
        d_k=d_k,
        d_v=d_v,
        d_ff=d_ff,
        base_regularizer=base_regularizer,
        end_regularizer=end_regularizer
    )

    generator.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='mse'
    )

    # ============================
    # Train
    # ============================
    trained_generator, best_val_accuracy = train_with_enantiomer_sampling(
        generator,
        dataG_train, labelsG_train, train_df,
        dataG_val, labelsG_val, val_df,
        epochs=1,
        batch_size=batch_size
    )

    # ============================
    # Evaluate
    # ============================
    test_accuracy = evaluate_ranking_accuracy(
        trained_generator,
        dataG_test, labelsG_test, test_df
    )

    print(f"Test Accuracy: {test_accuracy:.4f}")

    results.append({
        **params,
        'val_acc': best_val_accuracy,
        'test_acc': test_accuracy
    })

# ============================
# Best result
# ============================
best = max(results, key=lambda x: x['test_acc'])

print("\n===== BEST RESULT =====")
print(best)